[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C11_RAG_Retrieval_Course/04_rag_evaluation/04_rag_evaluation.ipynb)

# 04 · RAG 评测（纯 numpy/pandas）

目标：把 **faithfulness（忠实度/防幻觉）、answer relevance（切题）、context precision/recall（检索段质量）、RAGAS 四项合一、幻觉检测、归因诊断矩阵** 全部从零实现，并用 `assert` 验证。

路线：论断分解+faithfulness → answer relevance → context precision → context recall → RAGAS 四项合一(pandas 表) → 四象限诊断 → ✏️ 练习 → 📖 答案 → 🧪 真实数据(SQuAD)胶囊。

> 心智模型：RAG 评测 = **分科体检**。一次调用是三元组 (问题 q, 检索上下文 c, 答案 a)；我们用**规则版**(词重叠/集合包含作蕴含代理)模拟 RAGAS 的 LLM 判官，先看清每个指标*在量什么*。

## 1 · 论断分解 + faithfulness（防幻觉）

**faithfulness** = 答案里被上下文支持的论断比例。两步：① 把答案拆成原子论断；② 每条论断判是否被某条上下文「支持」。

真实 RAGAS 用 LLM 判蕴含；我们用**词重叠代理**：论断的内容词若有足够比例出现在某条上下文里，就算被支持。
造一个含 1 条**捏造论断**的答案，看 faithfulness 从 1.0 掉下来，并定位是哪条在编。

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)

STOP = {'the','a','an','is','are','was','were','of','in','on','at','to','and','or','it',
        'that','this','for','as','by','with','about','约','是','的','了','在','和'}

def content_words(text):
    '''小写分词、去停用词、去标点，返回内容词集合。'''
    toks = text.lower().replace(',', ' ').replace('.', ' ').replace(';', ' ').split()
    return {t for t in toks if t and t not in STOP}

def is_supported(claim, contexts, thresh=0.6):
    '''论断是否被【某条】上下文支持。两条任一成立即支持：
       (a) 论断作为子串出现在某条上下文里(抽取式答案的真实定义, 最强证据); 或
       (b) 论断内容词被某条上下文覆盖的比例 >= thresh(蕴含代理)。'''
    cl = claim.lower().strip()
    for ctx in contexts:
        if cl and cl in ctx.lower():    # (a) 子串包含 -> 直接被支持
            return True
    cw = content_words(claim)
    if not cw:
        return True                     # 空论断视为平凡成立
    best = 0.0
    for ctx in contexts:
        ctx_w = content_words(ctx)
        overlap = len(cw & ctx_w) / len(cw)
        best = max(best, overlap)
    return best >= thresh

def faithfulness(claims, contexts, thresh=0.6):
    '''被支持论断的比例 in [0,1]。'''
    if not claims:
        return 1.0
    supported = [is_supported(cl, contexts, thresh) for cl in claims]
    return sum(supported) / len(claims), supported

contexts = [
    'Paris is the capital and most populous city of France.',
    'Paris is located on the Seine river in northern France.',
]
# 忠实答案：每条论断都能在上下文找到依据
claims_good = ['Paris is the capital of France', 'Paris is on the Seine river']
# 含幻觉的答案：第2条数字是编的(上下文无人口数据)
claims_bad = ['Paris is the capital of France', 'Paris has population 50 million people']

f_good, sup_good = faithfulness(claims_good, contexts)
f_bad, sup_bad = faithfulness(claims_bad, contexts)
print(f'忠实答案 faithfulness = {f_good:.2f}  逐条支持: {sup_good}')
print(f'幻觉答案 faithfulness = {f_bad:.2f}  逐条支持: {sup_bad}')
print(f'  -> 疑似幻觉论断: {[c for c,s in zip(claims_bad, sup_bad) if not s]}')
assert f_good == 1.0, '全部有据 -> 忠实度 1.0'
assert f_bad < 1.0, '含 1 条捏造论断 -> 忠实度应下降'
assert sup_bad == [True, False], '应精确定位第 2 条为幻觉'
print('✅ faithfulness 正确：全有据=1.0，捏造论断拉低分数并被定位')

## 2 · answer relevance（切题，与 faithfulness 正交）

**answer relevance** = 答案是否切题回应问题，与 faithfulness 正交：可以忠实却答非所问。

代理：答案对**问题关键词的覆盖度**（真实 RAGAS 用反向生成问题再比相似度）。
对照「正面回答」vs「忠实但跑题」的答案——后者 relevance 应更低。

In [ ]:
def answer_relevance(question, answer):
    '''答案对问题内容词的覆盖比例，作为切题度代理 in [0,1]。'''
    qw = content_words(question)
    aw = content_words(answer)
    if not qw:
        return 0.0
    return len(qw & aw) / len(qw)

question = 'How tall is the Eiffel Tower in meters'
ans_ontopic = 'The Eiffel Tower is about 330 meters tall'         # 正面回答高度
ans_offtopic = 'The Eiffel Tower is located in Paris France'       # 忠实但跑题(没说高度)

r_on = answer_relevance(question, ans_ontopic)
r_off = answer_relevance(question, ans_offtopic)
print(f'切题答案 relevance  = {r_on:.2f}')
print(f'跑题答案 relevance  = {r_off:.2f}')
assert r_on > r_off, '正面回答应比跑题答案更切题'
print('✅ answer relevance 正确：切题 > 跑题（与是否有据无关，正交于 faithfulness）')

## 3 · context precision（检索上下文干不干净）

**context precision** = 检索到的上下文里，真正与问题相关的比例。低精度 = 带回一堆噪声片段。

判每条上下文是否与问题相关（词重叠代理），相关条数 / 总条数。
造一个混入 2 条无关片段的检索结果，看精度下降。

In [ ]:
def is_relevant_ctx(question, ctx, thresh=0.2):
    '''上下文是否与问题相关：问题内容词被该上下文覆盖比例 >= thresh。'''
    qw = content_words(question)
    if not qw:
        return False
    overlap = len(qw & content_words(ctx)) / len(qw)
    return overlap >= thresh

def context_precision(question, contexts, thresh=0.2):
    '''相关上下文条数 / 总条数 in [0,1]。'''
    if not contexts:
        return 0.0
    rel = [is_relevant_ctx(question, ctx, thresh) for ctx in contexts]
    return sum(rel) / len(contexts), rel

q = 'What is the capital of France'
# 干净检索：两条都相关
ctx_clean = ['Paris is the capital of France',
             'The capital city France is Paris on the Seine']
# 带噪检索：混入 2 条无关
ctx_noisy = ['Paris is the capital of France',
             'Bananas are a popular yellow fruit',
             'The stock market fell sharply today',
             'The capital city France is Paris']

p_clean, rel_clean = context_precision(q, ctx_clean)
p_noisy, rel_noisy = context_precision(q, ctx_noisy)
print(f'干净检索 precision = {p_clean:.2f}  逐条相关: {rel_clean}')
print(f'带噪检索 precision = {p_noisy:.2f}  逐条相关: {rel_noisy}')
assert p_clean == 1.0, '全相关 -> precision 1.0'
assert p_noisy < p_clean, '混入噪声 -> precision 下降'
print('✅ context precision 正确：噪声片段拉低检索精度')

## 4 · context recall（关键证据漏没漏 —— RAG 最常见失败根因）

**context recall** = 回答所需的要点中，被检索上下文覆盖的比例。低召回 = 关键证据没检索到，生成器无米之炊。

需要 ground truth：回答这个问题**到底需要哪些要点**(可由标准答案分解)。
看每个要点是否被某条上下文覆盖。造一个漏掉关键证据的检索，看 recall 下降。

In [ ]:
def point_covered(point, contexts, thresh=0.6):
    '''某个所需要点是否被某条上下文覆盖（内容词覆盖比例 >= thresh）。'''
    return is_supported(point, contexts, thresh)   # 复用第1节的支持判定

def context_recall(ground_truth_points, contexts, thresh=0.6):
    '''被覆盖的所需要点 / 全部所需要点 in [0,1]。'''
    if not ground_truth_points:
        return 1.0
    covered = [point_covered(pt, contexts, thresh) for pt in ground_truth_points]
    return sum(covered) / len(ground_truth_points), covered

question = 'Who wrote Hamlet and when was it written'
# 回答所需的两个要点
gt_points = ['Hamlet was written by William Shakespeare',
             'Hamlet was written around 1600']
# 完整检索：两个要点都被覆盖
ctx_full = ['Hamlet is a tragedy written by William Shakespeare',
            'Shakespeare wrote Hamlet around 1600 in England']
# 漏检：只覆盖作者，漏了年份
ctx_missing = ['Hamlet is a tragedy written by William Shakespeare',
               'Hamlet is one of the most famous plays ever']

r_full, cov_full = context_recall(gt_points, ctx_full)
r_miss, cov_miss = context_recall(gt_points, ctx_missing)
print(f'完整检索 recall = {r_full:.2f}  逐点覆盖: {cov_full}')
print(f'漏检     recall = {r_miss:.2f}  逐点覆盖: {cov_miss}')
print(f'  -> 漏掉的要点: {[pt for pt,c in zip(gt_points, cov_miss) if not c]}')
assert r_full == 1.0, '全覆盖 -> recall 1.0'
assert r_miss < r_full, '漏掉关键证据 -> recall 下降'
print('✅ context recall 正确：漏检拉低召回并定位漏掉的要点（RAG 最常见失败根因）')

## 5 · RAGAS 四项合一（无参考自动评测）

把四项指标合成一个函数 `ragas_eval(q, contexts, answer_claims, gt_points)`，对一次 RAG 调用一次性给出**faithfulness / answer relevance / context precision / context recall** 四分，打印成 pandas 表。

这就是 RAGAS 的骨架：仅凭 (q, c, a)[+所需要点] 自动打分，无需人工标准答案。

In [ ]:
def ragas_eval(question, contexts, answer_claims, gt_points):
    '''规则版 RAGAS：返回四项分的 dict。answer = 论断列表。'''
    faith, _ = faithfulness(answer_claims, contexts)
    answer_text = ' '.join(answer_claims)
    rel = answer_relevance(question, answer_text)
    prec, _ = context_precision(question, contexts)
    rec, _ = context_recall(gt_points, contexts)
    return {'faithfulness': faith, 'answer_relevance': rel,
            'context_precision': prec, 'context_recall': rec}

# 一个健康的 RAG 调用
q = 'What is the capital of France'
contexts = ['Paris is the capital of France',
            'Paris is the largest city in France on the Seine']
answer_claims = ['Paris is the capital of France']
gt_points = ['The capital of France is Paris']

scores = ragas_eval(q, contexts, answer_claims, gt_points)
df = pd.DataFrame([scores]).T.rename(columns={0: 'score'})
print(df.to_string())
assert all(0.0 <= v <= 1.0 for v in scores.values()), '四项分都应在[0,1]'
assert scores['faithfulness'] == 1.0 and scores['context_recall'] == 1.0
print('\n✅ RAGAS 四项合一：仅凭 (q,c,a)+所需要点自动给出四分，无需人工标准答案')

## 6 · 归因诊断：四象限定位瓶颈

给四个**不同病症**的 RAG 案例，用四项分定位每个的瓶颈：
- **完美**：四项都高。
- **漏检**(检索差)：context recall 低。
- **噪声**(检索差)：context precision 低。
- **幻觉**(生成差)：faithfulness 低。

验证每种病在**对应指标**上得分明显更低——这就是把『系统不好』拆成『哪一科几分』。

In [ ]:
q = 'Who painted the Mona Lisa'
gt = ['The Mona Lisa was painted by Leonardo da Vinci']

cases = {
    '完美': dict(
        contexts=['The Mona Lisa was painted by Leonardo da Vinci',
                  'Leonardo da Vinci painted the Mona Lisa in Italy'],
        claims=['The Mona Lisa was painted by Leonardo da Vinci']),
    '漏检': dict(   # 上下文没有作者信息 -> recall 低
        contexts=['The Mona Lisa is a famous portrait painting',
                  'The Mona Lisa hangs in the Louvre museum Paris'],
        claims=['The Mona Lisa is a famous painting']),
    '噪声': dict(   # 混入大量无关 -> precision 低
        contexts=['The Mona Lisa was painted by Leonardo da Vinci',
                  'Bananas are yellow tropical fruit',
                  'The weather today is sunny and warm',
                  'Football is a popular sport worldwide'],
        claims=['The Mona Lisa was painted by Leonardo da Vinci']),
    '幻觉': dict(   # 上下文有作者，但答案编了个错的 -> faithfulness 低
        contexts=['The Mona Lisa was painted by Leonardo da Vinci',
                  'Leonardo da Vinci painted the Mona Lisa in Italy'],
        claims=['The Mona Lisa was painted by Pablo Picasso artist']),
}

rows = []
for name, case in cases.items():
    s = ragas_eval(q, case['contexts'], case['claims'], gt)
    rows.append({'案例': name, **{k: round(v, 2) for k, v in s.items()}})
diag = pd.DataFrame(rows).set_index('案例')
print(diag.to_string())

# 验证每种病在对应指标上明显更低
assert diag.loc['漏检', 'context_recall'] < diag.loc['完美', 'context_recall'], '漏检->recall低'
assert diag.loc['噪声', 'context_precision'] < diag.loc['完美', 'context_precision'], '噪声->precision低'
assert diag.loc['幻觉', 'faithfulness'] < diag.loc['完美', 'faithfulness'], '幻觉->faithfulness低'
print('\n✅ 归因诊断：每种病在对应指标上得分最低 -> 据此定位修检索还是修生成')

---
## ✏️ 练习 1：faithfulness 打分

实现 `faithfulness_score(claims, contexts, thresh=0.6)`：返回被上下文支持的论断**比例**（只返回 float，不返回逐条列表）。

用蕴含代理：论断内容词被某条上下文覆盖比例 >= thresh 即算被支持。复用 `content_words`。

In [ ]:
def faithfulness_score(claims, contexts, thresh=0.6):
    # TODO: 对每条论断判是否被某条上下文支持(内容词覆盖>=thresh)，返回被支持比例
    #       空 claims 返回 1.0
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ctx = ['Mount Everest is the tallest mountain on Earth',
       'Mount Everest is located in the Himalayas Nepal']
all_good = ['Everest is the tallest mountain', 'Everest is in the Himalayas']
one_bad = ['Everest is the tallest mountain', 'Everest is 50000 meters tall fake']
assert faithfulness_score(all_good, ctx) == 1.0, '全有据=1.0'
assert faithfulness_score(one_bad, ctx) == 0.5, '1/2 有据=0.5'
assert faithfulness_score([], ctx) == 1.0, '空论断=1.0'
print('✅ 练习 1 通过：faithfulness 打分正确')

## ✏️ 练习 2：answer relevance

实现 `answer_relevance_score(question, answer)`：返回答案对问题内容词的覆盖比例 in [0,1]。

验证切题答案 > 跑题答案，且完全无关答案接近 0。

In [ ]:
def answer_relevance_score(question, answer):
    # TODO: 问题内容词被答案覆盖的比例；空问题返回 0.0
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
q = 'When did World War Two end'
on = 'World War Two ended in 1945'
off = 'Bananas grow in tropical climates'
r_on = answer_relevance_score(q, on)
r_off = answer_relevance_score(q, off)
assert r_on > r_off, '切题 > 跑题'
assert r_off < 0.2, '完全无关答案 relevance 接近 0'
assert 0.0 <= r_on <= 1.0
print(f'切题={r_on:.2f}  跑题={r_off:.2f}')
print('✅ 练习 2 通过：answer relevance 正确')

## ✏️ 练习 3：context recall

实现 `context_recall_score(ground_truth_points, contexts, thresh=0.6)`：返回被检索上下文覆盖的所需要点**比例** in [0,1]（只返回 float）。这是 RAG 最常用来诊断「漏检」的指标。

In [ ]:
def context_recall_score(ground_truth_points, contexts, thresh=0.6):
    # TODO: 每个所需要点是否被某条上下文覆盖(内容词覆盖>=thresh)，返回覆盖比例
    #       空要点返回 1.0
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
pts = ['The Great Wall is in China', 'The Great Wall is thousands of kilometers long']
full = ['The Great Wall is located in China',
        'The Great Wall stretches thousands of kilometers across China']
miss = ['The Great Wall is located in China',
        'The Great Wall is a famous tourist attraction']
assert context_recall_score(pts, full) == 1.0, '全覆盖=1.0'
assert context_recall_score(pts, miss) == 0.5, '漏1个=0.5'
assert context_recall_score([], miss) == 1.0
print('✅ 练习 3 通过：context recall 正确')

## ✏️ 练习 4：端到端评测

实现 `end_to_end_eval(question, contexts, answer_claims, gt_points)`：返回含四项分的 dict，键为`faithfulness / answer_relevance / context_precision / context_recall`。复用前面的实现。

这是一次 RAG 调用的完整体检报告。

In [ ]:
def end_to_end_eval(question, contexts, answer_claims, gt_points):
    # TODO: 调用四个指标，返回 dict（四个键如上）
    #   answer_relevance 用 ' '.join(answer_claims) 当答案文本
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
q = 'What is the speed of light'
ctx = ['The speed of light is about 300000 kilometers per second',
       'Light travels at approximately 300000 km per second in vacuum']
claims = ['The speed of light is about 300000 kilometers per second']
gt = ['The speed of light is 300000 km per second']
rep = end_to_end_eval(q, ctx, claims, gt)
assert set(rep.keys()) == {'faithfulness','answer_relevance','context_precision','context_recall'}
assert all(0.0 <= v <= 1.0 for v in rep.values())
assert rep['faithfulness'] == 1.0 and rep['context_recall'] == 1.0
print('体检报告:', {k: round(v,2) for k,v in rep.items()})
print('✅ 练习 4 通过：端到端评测给出完整四项体检报告')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def faithfulness_score(claims, contexts, thresh=0.6):
    if not claims:
        return 1.0
    supported = [is_supported(cl, contexts, thresh) for cl in claims]
    return sum(supported) / len(claims)

In [ ]:
# 练习 2 参考答案
def answer_relevance_score(question, answer):
    qw = content_words(question)
    if not qw:
        return 0.0
    return len(qw & content_words(answer)) / len(qw)

In [ ]:
# 练习 3 参考答案
def context_recall_score(ground_truth_points, contexts, thresh=0.6):
    if not ground_truth_points:
        return 1.0
    covered = [is_supported(pt, contexts, thresh) for pt in ground_truth_points]
    return sum(covered) / len(ground_truth_points)

In [ ]:
# 练习 4 参考答案
def end_to_end_eval(question, contexts, answer_claims, gt_points):
    answer_text = ' '.join(answer_claims)
    return {
        'faithfulness': faithfulness_score(answer_claims, contexts),
        'answer_relevance': answer_relevance_score(question, answer_text),
        'context_precision': context_precision(question, contexts)[0],
        'context_recall': context_recall_score(gt_points, contexts),
    }

---
## 🧪 真实数据胶囊：在 SQuAD 上评 RAG 的忠实度

用真实的 **SQuAD** 问答数据构造 RAG 三元组：(问题, 上下文段落, 答案=gold 答案跨度)。我们优先**联网下载**，失败则**回退到内置的真实 SQuAD 样本**，保证离线也能跑、逻辑一致。

任务：对比「**好答案**(=gold 答案跨度，必被上下文支持)」与「**幻觉答案**(编造内容)」的 faithfulness。

In [ ]:
# 优先联网拉真实 SQuAD；失败回退到内置真实样本（与模块 01 同款 loader）
def load_squad_samples(n=5):
    try:
        import urllib.request, json
        url = 'https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json'
        with urllib.request.urlopen(url, timeout=5) as f:
            data = json.load(f)
        out = []
        for art in data['data']:
            for para in art['paragraphs']:
                ctx = para['context']
                for qa in para['qas']:
                    if not qa.get('is_impossible', False) and qa['answers']:
                        out.append((qa['question'], ctx, qa['answers'][0]['text']))
                        break          # 本段落只取 1 个问题
                if len(out) >= n: break
            if len(out) >= n: break
        print(f'✅ 联网加载 SQuAD 成功，取 {len(out)} 条')
        return out
    except Exception as e:
        print(f'⚠ 联网失败({type(e).__name__})，回退内置真实 SQuAD 样本')
        return [
            ('In what country is Normandy located?',
             'The Normans were the people who in the 10th and 11th centuries gave their '
             'name to Normandy, a region in France. They were descended from Norse raiders.',
             'France'),
            ('When were the Normans in Normandy?',
             'The Normans were the people who in the 10th and 11th centuries gave their '
             'name to Normandy, a region in France.',
             '10th and 11th centuries'),
            ('What is the capital of France?',
             'Paris is the capital and most populous city of France, situated on the Seine.',
             'Paris'),
            ('What language did the Normans speak?',
             'The Norman dynasty had a major political and cultural impact; they spoke a '
             'language that evolved into Norman French.',
             'Norman French'),
            ('Photosynthesis occurs in which organelle?',
             'In plants, photosynthesis takes place in chloroplasts, which contain the '
             'pigment chlorophyll that captures light energy.',
             'chloroplasts'),
        ]

samples = load_squad_samples(5)
print(f'\n构造了 {len(samples)} 个 RAG 三元组 (问题/上下文/gold答案)')
print('例: Q =', samples[0][0][:55])
print('    A(gold) =', samples[0][2])

**🧪 胶囊练习**：实现 `faithfulness_on_squad(samples)`：对每个样本，把上下文当检索上下文、**gold 答案当『好答案』论断**，算 faithfulness，返回**平均忠实度**。gold 答案是从上下文里抽的跨度，理应被上下文高度支持 → 平均忠实度应很高。

In [ ]:
def faithfulness_on_squad(samples, thresh=0.6):
    # TODO: 对每个 (q, ctx, gold) 样本:
    #   claims=[gold], contexts=[ctx]，算 faithfulness_score，累加
    #   返回平均忠实度
    raise NotImplementedError

In [ ]:
# 自测
avg_faith = faithfulness_on_squad(samples)
print(f'gold 答案的平均 faithfulness = {avg_faith:.2f}')
# gold 答案是从上下文抽取的 -> 应被上下文高度支持
assert avg_faith >= 0.8, 'gold(抽取式)答案理应被上下文高度支持'
# 对照：一个明显编造的答案，faithfulness 应低得多
halluc = faithfulness(['The answer is a purple flying elephant xyz123'],
                       [samples[0][1]])[0]
assert halluc < avg_faith, '幻觉答案应远低于 gold 答案的忠实度'
print(f'对照-幻觉答案 faithfulness = {halluc:.2f}')
print('✅ 胶囊练习通过：真实 SQuAD 上 gold 答案高忠实、幻觉答案低忠实')

In [ ]:
# 📖 胶囊参考答案
def faithfulness_on_squad(samples, thresh=0.6):
    scores = []
    for q, ctx, gold in samples:
        scores.append(faithfulness_score([gold], [ctx], thresh))
    return float(np.mean(scores))

### 小结
- RAG 评测 = **分科体检**：一次调用是三元组 (问题 q, 检索上下文 c, 答案 a)，分别评检索段与生成段。
- **faithfulness**(c→a)：答案论断被上下文支持的比例；**忠实≠正确**，靠参数记忆蒙对也算不忠实(防幻觉的核心)。
- **answer relevance**(q→a)：答案切题度，与 faithfulness **正交**(可忠实却跑题)。
- **context precision**(c 干净) / **context recall**(c 全)：检索段质量；**recall 低=漏检=RAG 最常见失败根因**(且常逼出幻觉)。
- **RAGAS**：仅凭 (q,c,a) 的无参考四项自动评测；本课用规则版(词重叠/蕴含代理)模拟其 LLM 判官。
- **归因诊断矩阵**(检索×生成四象限)：把『RAG 不好』拆成『哪一科几分』-> 定位修检索还是修生成。

下一站：**模块 05 · 检索指标** —— 把检索质量的度量做严：P@k / R@k / MRR / MAP / nDCG 从零推导。